# Ordered Logistic Regression for Adoption Predictors Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the FAIR² dataset using the `mlcroissant` library, referencing all entities (record sets, fields, columns) by their `@id` for clarity and traceability.

### Dataset Source
The dataset is defined by a Croissant schema and is accessible at the following URL:

`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`

In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading
Load dataset metadata and records using `mlcroissant`. All entities will be referenced by their full `@id`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = "https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"Name: {metadata.name}\nDescription: {metadata.description}\n")

## 2. Data Overview
Inspect the available record sets (`cr:RecordSet`), and for each, list its fields and columns, using their exact `@id` values.

In [ ]:
# List all record sets and their field ids
from mlcroissant.dataset.metadata.record_set import RecordSet

record_sets = []
# The record sets list may be under metadata.record_sets or metadata.recordSet
if hasattr(metadata, "record_sets"):
    record_sets = metadata.record_sets
elif hasattr(metadata, "recordSet"):
    record_sets = metadata.recordSet

if not record_sets:
    print("No record sets found in metadata.\nCheck the Croissant schema if record sets are defined.")
else:
    print(f"Found {len(record_sets)} record sets:")
    for rs in record_sets:
        print(f"- RecordSet @id: {rs['@id'] if isinstance(rs, dict) and '@id' in rs else getattr(rs, '@id', None)}\n  Fields:")
        fields = []
        if isinstance(rs, dict) and 'field' in rs:
            fields = rs['field']
        elif hasattr(rs, 'field'):
            fields = rs.field
        for field in fields:
            if isinstance(field, dict):
                field_id = field.get('@id', None)
            else:
                field_id = getattr(field, '@id', None)
            print(f"    - field @id: {field_id}")

## 3. Data Extraction
Load data from a selected record set into a DataFrame using its full `@id`. All fields/columns referenced should also be by `@id`.

**Note:** If you have multiple record sets, you can loop and load all. Here, we'll attempt to load the first available record set.

In [ ]:
# If no record sets found, this cell will have nothing to load.
import warnings
dataframes = {}
loaded_record_set_id = None

if not record_sets:
    print("No record sets were found in the Croissant metadata; data extraction cannot proceed.")
else:
    # Use the first record set's @id
    first_rs = record_sets[0]
    if isinstance(first_rs, dict):
        record_set_id = first_rs.get("@id")
    else:
        record_set_id = getattr(first_rs, "@id", None)
    loaded_record_set_id = record_set_id
    print(f"Loading records for record set @id: {record_set_id}")

    # Load records into a DataFrame
    try:
        records = list(dataset.records(record_set=record_set_id))
        if not records:
            print(f"No records found for record set: {record_set_id}")
        else:
            df = pd.DataFrame(records)
            dataframes[record_set_id] = df
            print(f"Fields (DataFrame columns) in record set {record_set_id}:")
            print(df.columns.tolist())
            display(df.head())
    except Exception as e:
        warnings.warn(str(e))
        print(f"Could not load data for record set {record_set_id}. Error: {e}")

## 4. Exploratory Data Analysis (EDA)
Demonstrate data transformation steps: filtering by a numeric field, normalizing, and grouping. You'll need to specify the desired fields, by their full `@id` (use the output from the previous code if unsure).

If the data is empty, the cell will provide an example with synthetic data.

In [ ]:
import numpy as np
if not dataframes or not loaded_record_set_id or dataframes[loaded_record_set_id].empty:
    print("No data loaded for EDA. Generating example DataFrame.")
    # Example synthetic fields with IDs as would be shown in a real schema
    numeric_field_id = "http://senscience.ai/field/log_likelihood"
    group_field_id = "http://senscience.ai/field/model_type"
    df = pd.DataFrame({
        numeric_field_id: np.random.randn(100) * 10 + 50,
        group_field_id: np.random.choice(['indigenous', 'modern'], size=100)
    })
else:
    df = dataframes[loaded_record_set_id]
    # Attempt to pick numeric and grouping fields using their @id as column names
    # You should adjust these according to the printed column list if necessary
    # Example:
    # numeric_field_id = 'log_likelihood@id'
    # group_field_id = 'model_type@id'
    cols = df.columns.tolist()
    # Try to find candidate numeric and string columns
    numeric_field_id = None
    group_field_id = None
    for c in cols:
        if numeric_field_id is None and ("log" in c.lower() or "coeff" in c.lower() or df[c].dtype.kind in ('i','f')):
            numeric_field_id = c
        if group_field_id is None and ("type" in c.lower() or "group" in c.lower() or df[c].dtype=='object'):
            group_field_id = c
    if not numeric_field_id:
        numeric_field_id = cols[0]
    if not group_field_id:
        group_field_id = cols[1] if len(cols) > 1 else cols[0]

print(f"Using numeric field @id: {numeric_field_id}")
print(f"Using group field @id: {group_field_id}")

# Filtering
threshold = 10
filtered_df = df[df[numeric_field_id] > threshold]
print(f"Filtered records in {numeric_field_id} > {threshold}:")
display(filtered_df.head())

# Normalization
filtered_df = filtered_df.copy()
filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"Normalized {numeric_field_id} (z-score):")
display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Grouping
if group_field_id in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame("mean").join(
        filtered_df.groupby(group_field_id)[numeric_field_id].count().to_frame("count"))
    print(f"Grouped results by {group_field_id}:")
    display(grouped_df.head())

## 5. Visualization
Plot data distributions or relationships between fields using their `@id`.

Here, we visualize the normalized numeric field and group differences.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

if not filtered_df.empty:
    plt.figure(figsize=(8, 5))
    sns.histplot(filtered_df[f"{numeric_field_id}_normalized"], bins=20, kde=True)
    plt.title(f'Distribution of Normalized {numeric_field_id}')
    plt.xlabel(f'Normalized {numeric_field_id} (@id)')
    plt.ylabel('Count')
    plt.show()
    
    if group_field_id in filtered_df.columns:
        plt.figure(figsize=(8, 5))
        sns.boxplot(x=filtered_df[group_field_id], y=filtered_df[numeric_field_id])
        plt.title(f'{numeric_field_id} by {group_field_id}')
        plt.xlabel(f'{group_field_id} (@id)')
        plt.ylabel(f'{numeric_field_id}')
        plt.show()
else:
    print("No data for visualization.")

## 6. Conclusion
This notebook demonstrated how to load, inspect, and analyze a Croissant-specified dataset using the `mlcroissant` library while maintaining traceability via entity `@id` references. You can adapt this approach to your own FAIR datasets, always relying on the unique identifiers for robust, automated workflows.